# Specialists baseline — zero-shot OS-Atlas-4B and ShowUI-2B

**Owner:** Shared (this notebook produces H1's specialist column).  
**Goal:** Measure zero-shot specialist accuracy on the three benchmarks so Task 1's LoRA results have a comparison reference.

**Models**
- OS-Atlas-4B — `OS-Copilot/OS-Atlas-Pro-4B`
- ShowUI-2B — `showlab/ShowUI-2B`
- Ferret-UI Lite-3B has no public weights as of 2026-05-20; the published numbers (V2 91.6%, Pro 53.3%) are added as a reference row in the pivot below.

**Benchmarks**
- ScreenSpot-V2 (1272), ScreenSpot-Pro (1581), OSWorld-G (564).

Results stream to `results/eval/specialists_baseline.jsonl` row by row, so a Colab disconnect doesn't lose accumulated progress.


In [ ]:
# Colab bootstrap. On local Jupyter this is a no-op because the env is already set up.
import sys
from pathlib import Path

REPO_URL = "https://github.com/ali-epita/action-learning-ais5.git"
REPO_DIR = "/content/action-learning-ais5"

if "google.colab" in sys.modules:
    if not Path(REPO_DIR).exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    # --no-deps preserves Colab's pre-bundled torch + CUDA build.
    !pip install -q -e . --no-deps

    # transformers<5 because the Qwen2.5-VL wrapper targets 4.49+. peft for LoRA.
    # Strip Colab's pre-installed torchao (peft 0.17+ raises ImportError on the
    # 0.10.0 that ships with Colab instead of treating it as absent). LoRA
    # fine-tuning here does not depend on torchao.
    !pip uninstall -y -q torchao
    !pip install -q "transformers>=4.49,<5" "accelerate>=0.34" "datasets>=3.0" \
                    "peft>=0.13" "qwen-vl-utils>=0.0.10" "bitsandbytes>=0.44" \
                    rich tqdm pyyaml typer pillow huggingface_hub safetensors psutil

    src_dir = str(Path(REPO_DIR) / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import importlib
    importlib.invalidate_caches()

import ais5
print(f"ais5 v{ais5.__version__} ready  (colab={'google.colab' in sys.modules})")


In [ ]:
import json
from pathlib import Path

import pandas as pd

from ais5.data import load_benchmark
from ais5.eval import evaluate_model
from ais5.eval.breakdown import by_target_type, by_ui_type
from ais5.models import get_model
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)


In [ ]:
# Smoke vs full toggle. SMOKE_MODE keeps the run cheap so you can validate
# end-to-end behavior in minutes; the full grid is what produces H1's
# specialist comparison column.
SMOKE_MODE = True

if SMOKE_MODE:
    MODELS  = ['showui-2b']                   # smallest specialist, fastest sanity check
    BENCHES = ['screenspot-v2']
    LIMIT   = 50
else:
    MODELS  = ['os-atlas-4b', 'showui-2b']
    BENCHES = ['screenspot-v2', 'screenspot-pro', 'osworld-g']
    LIMIT   = None                             # None = full benchmark

results_dir = Path('results/eval')
results_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = results_dir / 'specialists_baseline.jsonl'
jsonl_path.unlink(missing_ok=True)

print(f"SMOKE_MODE={SMOKE_MODE}  models={MODELS}  benches={BENCHES}  limit={LIMIT}")


## Main eval loop


In [ ]:
import torch

# Cache samples per benchmark so each model only triggers one dataset load.
samples_by_bench = {b: list(load_benchmark(b)) for b in BENCHES}

eval_runs = {}
rows = []

for model_name in MODELS:
    print(f"\n=== {model_name} ===")
    try:
        model = get_model(model_name)
    except Exception as e:
        print(f"SKIP {model_name}: {e}")
        continue

    for bench in BENCHES:
        run = evaluate_model(model, samples_by_bench[bench], benchmark=bench, limit=LIMIT)
        eval_runs[(model_name, bench)] = run
        n = len(run.results)
        parsed = sum(1 for r in run.results if r.pred is not None)
        row = {
            'model': model_name,
            'benchmark': bench,
            'accuracy': run.accuracy,
            'n_samples': n,
            'parsed_rate': parsed / max(1, n),
        }
        rows.append(row)
        with jsonl_path.open('a') as f:
            f.write(json.dumps(row) + '\n')
        print(f"  {bench:18s} acc={run.accuracy:.4f}  parsed={parsed/n:.1%}  (n={n})")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(results_dir / 'specialists_baseline.csv', index=False)
df


## Specialist accuracy pivot

Side-by-side accuracy on each benchmark per specialist, with Ferret-UI Lite's published numbers as a reference.


In [ ]:
# Pivot for the H1 comparison table.
pivot = df.pivot(index='model', columns='benchmark', values='accuracy')

# Ferret-UI Lite-3B has no public weights as of 2026-05-20; cite the paper's
# published numbers as a reference row so the comparison table is complete.
FERRET_REF = {
    'screenspot-v2':  0.916,
    'screenspot-pro': 0.533,
    # OSWorld-G not in the Ferret-UI Lite paper — left as NaN.
}
pivot.loc['ferret-ui-lite-3b (paper)'] = pd.Series(FERRET_REF)
pivot


## Per-UI breakdown


In [ ]:
# How specialists distribute errors across UI categories. Useful for H1's
# error analysis: if a specialist struggles on the same UI buckets as the
# LoRA-adapted generalist, the gap is platform-side rather than model-side.
for model_name in MODELS:
    bench = BENCHES[0]
    key = (model_name, bench)
    if key not in eval_runs:
        continue
    print(f"\n--- {model_name} on {bench} ---")
    display(by_ui_type(eval_runs[key].results))


## Per-target-type breakdown (text vs icon)


In [ ]:
# Text vs icon. Specialists are usually trained heavily on icons; this
# breakdown shows whether the icon advantage is the biggest part of their gap.
for model_name in MODELS:
    bench = BENCHES[0]
    key = (model_name, bench)
    if key not in eval_runs:
        continue
    print(f"\n--- {model_name} on {bench} ---")
    display(by_target_type(eval_runs[key].results))
